### **Autoloader**

## **DimUser**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

import os 
import sys

project_pth = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)

from utils.transformations import reusable

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimUser/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@doristorageazureproject.dfs.core.windows.net/DimUser")

In [0]:
display(df_user,checkpointLocation = "abfss://silver@doristorageazureproject.dfs.core.windows.net/DimUser/displaycheckpoint")

Checkpointing to abfss://silver@doristorageazureproject.dfs.core.windows.net/DimUser/displaycheckpoint


In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])



In [0]:
df_user.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimUser/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimUser/data")\
            .toTable("spotify_cata.silver.DimUser")

In [0]:
print("Spark version:", spark.version)
print("Is streaming:", df_user.isStreaming)



Spark version: 4.2.0
Is streaming: True


### **DimArtist**




In [0]:
df_art = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimArt/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@doristorageazureproject.dfs.core.windows.net/DimArtist")

In [0]:

display(
    df_art,
    checkpointLocation="abfss://silver@doristorageazureproject.dfs.core.windows.net/DimArt/checkpoint"
)

Checkpointing to abfss://silver@doristorageazureproject.dfs.core.windows.net/DimArt/checkpoint


In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])

In [0]:
df_art

DataFrame[artist_id: int, artist_name: string, genre: string, country: string, updated_at: timestamp]

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "200")


In [0]:
print("Is streaming:", df_art.isStreaming)
df_art.explain(True)

Is streaming: True
== Parsed Logical Plan ==
'UnresolvedDeduplicate DeduplicateKeyColumns(List(artist_id)), false, false
+- ~Project [artist_id#10423, artist_name#10424, genre#10425, country#10426, updated_at#10427]
   +- StreamingRelation cloudFiles, [artist_id#10423, artist_name#10424, genre#10425, country#10426, updated_at#10427, _rescued_data#10428], name=<Unassigned>

== Analyzed Logical Plan ==
artist_id: int, artist_name: string, genre: string, country: string, updated_at: timestamp
~Deduplicate [artist_id#10423]
+- ~Project [artist_id#10423, artist_name#10424, genre#10425, country#10426, updated_at#10427]
   +- StreamingRelation cloudFiles, [artist_id#10423, artist_name#10424, genre#10425, country#10426, updated_at#10427, _rescued_data#10428], name=<Unassigned>

== Optimized Logical Plan ==
~Deduplicate [artist_id#10423]
+- ~Project [artist_id#10423, artist_name#10424, genre#10425, country#10426, updated_at#10427]
   +- StreamingRelation cloudFiles, [artist_id#10423, artist_nam

In [0]:
print(spark.conf.get("spark.sql.shuffle.partitions"))

200


In [0]:

query = (
    df_art.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", "abfss://silver@doristorageazureproject.dfs.core.windows.net/DimArt/write_checkpoint")
        .trigger(availableNow=True)
        .option("path", "abfss://silver@doristorageazureproject.dfs.core.windows.net/DimArt/data")
        .toTable("spotify_cata.silver.DimArtist")
)

query.awaitTermination()

## **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@doristorageazureproject.dfs.core.windows.net/DimTrack")

In [0]:

display(
    df_track,
    checkpointLocation="abfss://silver@doristorageazureproject.dfs.core.windows.net/DimTrack/checkpoint"
)

Checkpointing to abfss://silver@doristorageazureproject.dfs.core.windows.net/DimTrack/checkpoint


In [0]:
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace
)

In [0]:
import os
import sys

from pyspark.sql.functions import col, regexp_replace, when

project_pth = os.path.join(os.getcwd(), '..', '..')
if project_pth not in sys.path:
    sys.path.append(project_pth)

from utils.transformations import reusable

df_track = df_track.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
        .when(col("duration_sec") < 300, "medium")
        .otherwise("high")
)

df_track = df_track.withColumn(
    "track_name",
    regexp_replace(col("track_name"), "-", " ")
)

df_track = reusable().dropColumns(df_track, ["_rescued_data"])

In [0]:
df_track.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimTrack/checkpoint1")\
            .trigger(once=True)\
            .option("path","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotify_cata.silver.DimTrack")

%md
## **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimDate/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@doristorageazureproject.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])

df_date.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimDate/checkpoint1")\
            .trigger(once=True)\
            .option("path","abfss://silver@doristorageazureproject.dfs.core.windows.net/DimDate/data")\
            .toTable("spotify_cata.silver.DimDate")

%md
## **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/FactStream/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@doristorageazureproject.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact,checkpointLocation="abfss://silver@doristorageazureproject.dfs.core.windows.net/FactStream/checkpoint")

Checkpointing to abfss://silver@doristorageazureproject.dfs.core.windows.net/FactStream/checkpoint


In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@doristorageazureproject.dfs.core.windows.net/FactStream/checkpoint1")\
            .trigger(once=True)\
            .option("path","abfss://silver@doristorageazureproject.dfs.core.windows.net/FactStream/data")\
            .toTable("spotify_cata.silver.FactStream")